In [1]:
pip install --upgrade datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 55.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.31.0
    Uninstalling huggingface_hub-1.31.0:
      Successfully uninstalled huggingface_hub-1.31.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5


In [2]:
from datasets import load_dataset

In [3]:
ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:           |  0.00B            

...

wikitext-103-raw-v1/validation-00000-of-(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-103-raw-v1/validation-00000-of-(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [4]:
train_text = ds["train"]["text"]

In [5]:
for i in range(0, 2):
  print(train_text[i])


 = Valkyria Chronicles III = 



In [6]:
sentences = [s for s in train_text if len(s.strip()) > 0]

In [7]:
for i in range(0, 2):
  print(sentences[i])

 = Valkyria Chronicles III = 

 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . 



In [8]:
import random

def create_sentence_pairs(sentences):
  pairs = []
  for i in range(len(sentences) - 1):
    if random.random() < 0.5:
      pairs.append({
          "sentence_a": sentences[i],
          "sentence_b": sentences[i+1],
          "label":1
      })
    else:
      random_sentence = random.choice(sentences)
      pairs.append({
          "sentence_a": sentences[i],
          "sentence_b": random_sentence,
          "label": 0
      })
  return pairs

In [9]:
pairs = create_sentence_pairs(sentences[:20000])
pairs[0]

{'sentence_a': ' = Valkyria Chronicles III = \n',
 'sentence_b': " Fowler 's performance in the 1997 – 98 football season was marred by a knee ligament injury that kept him out of action for half the season and caused him to miss the 1998 World Cup . During this period of injury , fellow Liverpool striker , Michael Owen rose to prominence , making his debut in 1997 . Owen established himself in the Liverpool team in Fowler 's absence . \n",
 'label': 0}

In [10]:
pairs[100]

{'sentence_a': ' = Cicely Mary Barker = \n',
 'sentence_b': " Cicely Mary Barker ( 28 June 1895 – 16 February 1973 ) was an English illustrator best known for a series of fantasy illustrations depicting fairies and flowers . Barker 's art education began in girlhood with correspondence courses and instruction at the Croydon School of Art . Her earliest professional work included greeting cards and juvenile magazine illustrations , and her first book , Flower Fairies of the Spring , was published in 1923 . Similar books were published in the following decades . \n",
 'label': 1}

In [11]:
len(pairs)

19999

In [12]:
type(pairs)

list

In [13]:
from datasets import Dataset

dataset = Dataset.from_list(pairs)
type(dataset)

datasets.arrow_dataset.Dataset

In [14]:
dataset[200]

{'sentence_a': " The font in St Andrew 's Church , South Croydon \n",
 'sentence_b': ' Out of Great Tribulation ; memorial chapel , Norbury Medthodist church , 1948 \n',
 'label': 1}

In [15]:
from transformers import BertTokenizerFast

In [16]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [17]:
def tokenize_function(example):
  return tokenizer(
      example["sentence_a"],
      example["sentence_b"],
      truncation=True, #Truncate a BERT model window case
      padding="max_length", #Matching a equal length
      max_length=128 #No of token
      )

In [18]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/19999 [00:00<?, ? examples/s]

In [19]:
tokenized_dataset[0]

{'sentence_a': ' = Valkyria Chronicles III = \n',
 'sentence_b': " Fowler 's performance in the 1997 – 98 football season was marred by a knee ligament injury that kept him out of action for half the season and caused him to miss the 1998 World Cup . During this period of injury , fellow Liverpool striker , Michael Owen rose to prominence , making his debut in 1997 . Owen established himself in the Liverpool team in Fowler 's absence . \n",
 'label': 0,
 'input_ids': [101,
  1027,
  11748,
  4801,
  4360,
  11906,
  3523,
  1027,
  102,
  14990,
  1005,
  1055,
  2836,
  1999,
  1996,
  2722,
  1516,
  5818,
  2374,
  2161,
  2001,
  24563,
  2011,
  1037,
  6181,
  25641,
  4544,
  2008,
  2921,
  2032,
  2041,
  1997,
  2895,
  2005,
  2431,
  1996,
  2161,
  1998,
  3303,
  2032,
  2000,
  3335,
  1996,
  2687,
  2088,
  2452,
  1012,
  2076,
  2023,
  2558,
  1997,
  4544,
  1010,
  3507,
  6220,
  11854,
  1010,
  2745,
  7291,
  3123,
  2000,
  12144,
  1010,
  2437,
  2010,
  28

In [20]:
tokenized_dataset[0].keys()

dict_keys(['sentence_a', 'sentence_b', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])

### **label**: The target ground-truth output for the model. For sequence classification, this is usually an integer representing a class (e.g., 0 for contradiction, 1 for entailment).

### **input_ids**: The numerical representations (IDs) of the tokens after the sentences are split and mapped to the model's vocabulary. This includes special tokens like [CLS] and [SEP].

### **token_type_ids**: Also known as segment IDs. It tells the model which tokens belong to sentence_a (usually marked with 0) and which belong to sentence_b (usually marked with 1).

### **attention_mask**: A binary mask that tells the model which tokens to pay attention to (1 for actual words) and which to ignore (0 for padding tokens added to make batch sizes uniform).


In [21]:
tokenized_dataset = tokenized_dataset.rename_column("label", "next_sentence_label")
tokenized_dataset = tokenized_dataset.remove_columns(["sentence_a", "sentence_b"])

In [22]:
tokenized_dataset[0].keys()

dict_keys(['next_sentence_label', 'input_ids', 'token_type_ids', 'attention_mask'])

In [23]:
tokenized_dataset.set_format("torch") #change return into pytorch

In [24]:
from transformers import DataCollatorForLanguageModeling, BertConfig, BertForPreTraining, TrainingArguments

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability = 0.15)

In [25]:
data_collator

DataCollatorForLanguageModeling(tokenizer=BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), mlm=True, whole_word_mask=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, return_ten

In [26]:
config = BertConfig(
    vocab_size=tokenizer.vocab_size, #Size of the vocabulary in this tokenizer
    hidden_size=256, #Vector dimension
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=512, #FeedForward layer parameter
    max_position_embeddings=128 #Context window of the model
)

In [27]:
model = BertForPreTraining(config)

In [28]:
training_args = TrainingArguments(
    output_dir = "./bert_mlm_nsp",
    per_device_train_batch_size = 16,
    num_train_epochs=10,
    logging_steps = 100,
    save_steps = 1000,
    learning_rate = 5e-4,
    weight_decay = 0.01,
    warmup_steps = 500,
    fp16 = True
)

In [29]:
from transformers import Trainer

trainer = Trainer(
    model = model,
    args = training_args,
    data_collator = data_collator,
    train_dataset = tokenized_dataset
)

In [30]:
trainer.train()

Step,Training Loss
100,10.544178
200,8.561288
300,8.031176
400,7.904570
500,7.786682
600,7.727359
700,7.608740
800,7.576335
900,7.524086
1000,7.501319


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=12500, training_loss=6.658692133789063, metrics={'train_runtime': 454.5143, 'train_samples_per_second': 440.008, 'train_steps_per_second': 27.502, 'total_flos': 348970966579200.0, 'train_loss': 6.658692133789063, 'epoch': 10.0})

In [31]:
trainer.save_model("./mini-bert-mlm-nsp-model")
tokenizer.save_pretrained("./mini-bert-mlm-nsp-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./mini-bert-mlm-nsp-model/tokenizer_config.json',
 './mini-bert-mlm-nsp-model/tokenizer.json')

In [32]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import whoami
whoami()

In [35]:
model.push_to_hub("asqar268008/mini-bert-mlm-nsp-model")
tokenizer.push_to_hub("asqar268008/mini-bert-mlm-nsp-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...fqqlz47/model.safetensors:   1%|1         |  559kB / 40.5MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/asqar268008/mini-bert-mlm-nsp-model/commit/c045b251d8a90d9218720158a36bf96f9e767ac2', commit_message='Upload tokenizer', commit_description='', oid='c045b251d8a90d9218720158a36bf96f9e767ac2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/asqar268008/mini-bert-mlm-nsp-model', endpoint='https://huggingface.co', repo_type='model', repo_id='asqar268008/mini-bert-mlm-nsp-model'), pr_revision=None, pr_num=None)